In [1]:
#%% ----- CELL 1 -----
# ============ NOTEBOOK 10a | CELL 1 : Load data, define RR and the "50-50" and "death" situations ============
# TECHNICAL: we reuse ball_master (every legal/illegal ball with expected runs, win probability and pressure flags).
# FUNCTIONAL: before recommending players, we must first find WHERE Rajasthan Royals actually lose value.
#             A gap in the death overs needs a death specialist; a gap in 50-50 moments needs a different profile.
import pandas as pd, numpy as np
import os

# find the interim folder that holds ball_master.parquet (searches the notebook folder and up to 2 levels above)
INTERIM = None
for cand in ["interim/", "../interim/", "../../interim/", "./", "../"]:
    if os.path.exists(cand + "ball_master.parquet"):
        INTERIM = cand; break
if INTERIM is None:
    INTERIM = "C:/PASTE/FULL/PATH/TO/interim/"        # <- set by hand (use forward slashes) if the search above fails
print("Using INTERIM =", os.path.abspath(INTERIM))
ball   = pd.read_parquet(INTERIM+"ball_master.parquet")
match  = pd.read_parquet(INTERIM+"matches_clean.parquet")
pm     = pd.read_parquet(INTERIM+"player_master.parquet")
squads = pd.read_parquet(INTERIM+"squads.parquet")

RR = 3                                    # Rajasthan Royals team_id in the data
# --- keep IPL matches only (league baselines must be IPL-vs-IPL, not international games) ---
ipl_ids = set(match.loc[match.comp_name.str.startswith("IPL"), "match_id"])
ib = ball[ball.match_id.isin(ipl_ids)].copy()
ib["season"] = ib.year

# --- situation flags ---
# 50-50 ball  : the batting side's win probability is between 40% and 60% (outcome still genuinely open)
# death ball  : overs 16-20 (the data's own phase definition), shown separately for innings 1 and 2
ib["fifty"] = ib.wp_before.between(0.40, 0.60)
ib["is_death"] = ib.phase.eq("death")
ib["fifty_death"] = ib.fifty & ib.is_death

print("IPL balls:", len(ib), "| seasons:", sorted(ib.season.unique()))
print("RR batting balls:", (ib.batting_team_id==RR).sum(), "| RR bowling balls:", (ib.bowling_team_id==RR).sum())
print("Share of balls that are 50-50:", round(ib.fifty.mean(),3), "| death:", round(ib.is_death.mean(),3),
      "| 50-50 AND death:", round(ib.fifty_death.mean(),3))

# --- RR current squad (IPL 2026 list) for later cells ---
rr_squad = squads[(squads.team_id==RR)&(squads.comp_name=="IPL 2026")][["player_id","player_name"]].drop_duplicates()
print("RR 2026 squad size in data:", len(rr_squad))


Using INTERIM = C:\Users\User\SupeRR Selector 2.0\interim
IPL balls: 87673 | seasons: [2022, 2023, 2024, 2025, 2026]
RR batting balls: 9101 | RR bowling balls: 9372
Share of balls that are 50-50: 0.301 | death: 0.229 | 50-50 AND death: 0.047
RR 2026 squad size in data: 27


**Cell 1: Setup and definitions**

**Procedure**
1. Loaded the ball-level data and kept **IPL matches only** (87,673 balls, 2022–2026), so RR is compared with the other IPL teams and not with international sides.
2. Defined two situations. A **50-50 ball** is one where the batting side's win probability is between 40% and 60%, so the outcome is still genuinely open. A **death ball** is one in overs 16–20 (the data's own phase definition).
3. Loaded the RR 2026 squad (27 players), used later to see who we already have.

**Observations**
- About **30%** of IPL balls are 50-50, **23%** are death balls, and only about **5%** are both.
- RR have 9,101 batting balls and 9,372 bowling balls in the sample.
- The 50-50 death slice is small for a single team (534 batting balls, 512 bowling balls), so any conclusion about that slice is fragile.

In [2]:
# ==== TABLE DISPLAY HELPER: paste once as a new cell right after cell 1 of the notebook, then run it ====
# Why: print(df.to_string()) dumps plain text that wraps and is hard to scan. A pandas "Styler" renders the same DataFrame as a real
# HTML table in Jupyter: coloured header, aligned numbers, shaded ratings, green/red verdicts. The data is NOT changed, only how it looks.
import numpy as np, pandas as pd
from IPython.display import display, HTML

_PINK, _BLUE = "#EA1A85", "#254AA5"
_GOOD, _BAD = "#E3F4EC", "#FBE4E4"          # soft green / soft red backgrounds for verdicts and yes/no flags

def showtbl(df, title=None, index=False, heat=None, dec=2, maxrows=None):
    """Render a DataFrame as a styled table.
    title   : heading shown above the table
    index   : True keeps the row labels (use for pivot tables and value counts); False hides them (player_id etc.)
    heat    : a matplotlib colormap name; shades EVERY number in the table (e.g. 'RdYlGn' for good-is-high, 'RdYlGn_r' for ranks)
    dec     : decimals shown (display only)
    maxrows : show only the first N rows
    Columns named rating / fit / fit_adj / gap / per100 are shaded automatically; 'verdict' and 'fits_*' columns get green/red cells."""
    d = df.copy()
    if isinstance(d, pd.Series): d = d.to_frame()
    if maxrows: d = d.head(maxrows)
    if isinstance(d.columns, pd.MultiIndex) or d.columns.name is not None: d.columns = [str(c) for c in d.columns]
    st = d.style
    if not index:
        try: st = st.hide(axis="index")
        except Exception: st = st.hide_index()                      # older pandas
    num = d.select_dtypes("number").columns
    st = st.format({c: ("{:,.0f}" if (d[c].dropna() % 1 == 0).all() and c != "age" else "{:,.%df}" % dec) for c in num}, na_rep="–")
    if heat:
        try: st = st.background_gradient(cmap=heat, axis=None, subset=list(num))
        except Exception: pass
    else:
        for c in d.columns:                                            # shade the headline score columns
            if c in ("rating","fit","fit_adj","gap","per100","raw_per100","raw_per24") and c in num and d[c].notna().any():
                st = st.background_gradient(subset=[c], cmap="RdPu")
    def _cell(v):                                                      # colour verdict words and yes/no flags
        s = str(v).lower()
        if s in ("real strength","yes","affordable"):          return f"background-color:{_GOOD};font-weight:600"
        if s in ("real weakness","no") or s.startswith("over"): return f"background-color:{_BAD};font-weight:600"
        return ""
    for c in d.columns:
        if c == "verdict" or str(c).startswith("fits_") or c == "price_layer":
            st = (st.map if hasattr(st, "map") else st.applymap)(_cell, subset=[c])
    st = st.set_table_styles([
        {"selector": "th", "props": [("background", _BLUE), ("color", "white"), ("font-weight", "600"), ("padding", "6px 10px"), ("text-align", "left")]},
        {"selector": "td", "props": [("padding", "5px 10px"), ("border-bottom", "1px solid #E8EBF2")]},
        {"selector": "tr:hover td", "props": [("background-color", "#F3F4FA")]}])
    if title: display(HTML(f'<div style="margin:14px 0 4px;font-size:15px;font-weight:700;color:{_BLUE}">{title}</div>'))
    display(st)
print("showtbl() ready")

showtbl() ready


In [3]:

#%% ----- CELL 2 -----
# ============ CELL 2 : Team-level diagnosis, RR vs the 9 other teams, by season and situation ============
# TECHNICAL: net_bat / net_bowl = runs above expected per ball, with wickets converted to runs (built in 05c).
#            We sum them over a situation and scale to "per 100 balls" (batting) or "per 24 balls" (bowling, 4 overs).
# FUNCTIONAL: "How many runs per 100 balls do RR gain or lose vs an average team facing the same situation?"
def team_table(df, side, mask, label):
    if side=="bat":
        g = df[mask & df.net_bat.notna()].groupby(["season","batting_team_id"]).agg(balls=("net_bat","size"), net=("net_bat","sum"))
        g["per"] = g.net/g.balls*100
    else:
        g = df[mask].groupby(["season","bowling_team_id"]).agg(balls=("net_bowl","size"), net=("net_bowl","sum"))
        g["per"] = g.net/g.balls*24
    g = g.reset_index()
    g.columns = ["season","team_id","balls","net","per"]
    g["rank"] = g.groupby("season").per.rank(ascending=False)       # 1 = best of 10
    out = g[g.team_id==RR][["season","balls","per","rank"]].copy()
    out.insert(0,"situation",label); out.insert(1,"side",side)
    return out

rows=[]
sit = {"Powerplay (1-6)": ib.phase.eq("powerplay"),
       "Middle (7-15)": ib.phase.eq("middle"),
       "Death (16-20)": ib.is_death,
       "50-50 (all overs)": ib.fifty,
       "Death in innings 1": ib.is_death & ib.innings.eq(1),
       "Death in innings 2": ib.is_death & ib.innings.eq(2),
       "50-50 AND death": ib.fifty_death,
       "Pressure (top 20% leverage)": ib.press_main.astype(bool) if "press_main" in ib else ib.bat_press}
for lab,mk in sit.items():
    for side in ["bat","bowl"]:
        rows.append(team_table(ib, side, mk, lab))
rr_diag = pd.concat(rows, ignore_index=True)
rr_diag["per"] = rr_diag.per.round(2)

# pooled across all five seasons
def pooled(side, mk):
    if side=="bat":
        g=ib[mk&ib.net_bat.notna()].groupby("batting_team_id").net_bat.agg(["size","sum"]); g["per"]=g["sum"]/g["size"]*100
    else:
        g=ib[mk].groupby("bowling_team_id").net_bowl.agg(["size","sum"]); g["per"]=g["sum"]/g["size"]*24
    return g.per.rank(ascending=False)[RR], g.per[RR], g["size"][RR]
pool=[]
for lab,mk in sit.items():
    for side in ["bat","bowl"]:
        r,p,n=pooled(side,mk); pool.append((lab,side,n,round(p,2),int(r)))
pool=pd.DataFrame(pool,columns=["situation","side","RR_balls","RR_net_per_unit","rank_of_10_pooled"])

print("Units: batting = net runs/100 balls ; bowling = net runs saved/24 balls. Positive = better than expected. Rank 1 = best of 10.\n")
showtbl(pool, "RR pooled 2022-26: net value and rank among 10 teams (rank 1 = best)")
showtbl(rr_diag.pivot_table(index=["situation","side"], columns="season", values="per"), "RR net value by season", index=True, heat="RdYlGn")
showtbl(rr_diag.pivot_table(index=["situation","side"], columns="season", values="rank"), "RR rank by season (1 = best)", index=True, heat="RdYlGn_r", dec=0)
rr_diag.to_pickle(INTERIM+"rr_diag_season.pkl"); pool.to_pickle(INTERIM+"rr_diag_pooled.pkl")



Units: batting = net runs/100 balls ; bowling = net runs saved/24 balls. Positive = better than expected. Rank 1 = best of 10.



situation,side,RR_balls,RR_net_per_unit,rank_of_10_pooled
Powerplay (1-6),bat,"2,749",17.17,1
Powerplay (1-6),bowl,"2,845",1.70,2
Middle (7-15),bat,"4,033",-5.53,10
Middle (7-15),bowl,"4,277",-1.22,9
Death (16-20),bat,"1,966",-2.15,7
Death (16-20),bowl,"2,250",-1.01,9
50-50 (all overs),bat,"2,823",8.36,2
50-50 (all overs),bowl,"2,956",0.78,1
Death in innings 1,bat,"1,134",-3.26,7
Death in innings 1,bowl,"1,194",-0.68,8


**Cell 2: RR's rank among 10 teams, by phase and season**

**Procedure**
1. For every ball, took **runs above expected** (wickets converted into runs, from notebook 05c).
2. Added these up per team and situation, scaled to **net runs per 100 balls** for batting and **net runs saved per 24 balls** for bowling. Positive means better than expected.
3. Ranked RR against the other nine teams for each season and pooled over 2022–26 (rank 1 = best).

**Observations**
- **Powerplay batting is elite:** 1st pooled (+16.6 per 100 balls), and 1st in both 2025 and 2026. This is the "RR start well" part of the theory.
- **Middle-overs (7–15) batting is the weakest area:** 10th pooled (−5.2), and 9th in both 2025 and 2026. Middle-overs bowling is 9th pooled (−1.4 per 24 balls).
- **Death overs are below average on both sides and have worsened.** Batting is 6th pooled but 8th in both 2025 and 2026. Bowling is 9th pooled, and innings-1 death bowling was 9th in both 2025 and 2026.
- **50-50 batting is strong:** 2nd pooled (+8.7). 50-50 bowling is average (4th, +0.04).
- **50-50 and death together:** batting is 4th, but bowling is 10th (−2.8) on a small sample.
- **Pressure (top 20% leverage):** batting is 3rd (+11.1) and bowling is 3rd (+0.7, barely above average).
- Season-to-season ranks swing a lot (for example, 50-50 batting went 2, 4, 7, 2, 5), which is why Cell 3 tests whether the gaps are real.

In [4]:
#%% ----- CELL 3 -----
# ============ CELL 3 : How sure are we? Bootstrap the RR gaps by match ============
# TECHNICAL: resample RR's matches with replacement (2,000 times); each match is one unit because balls inside a match are correlated.
#            The league-average per-ball value is ~0 by construction, so RR's own average is its gap vs league.
# FUNCTIONAL: A weak rank over 5 seasons could still be bad luck. The 80% band tells us if a weakness is real or noise.
rng = np.random.default_rng(7)
def boot(side, mk, B=2000):
    col, tid, k = ("net_bat","batting_team_id",100) if side=="bat" else ("net_bowl","bowling_team_id",24)
    d = ib[mk & (ib[tid]==RR) & ib[col].notna()].groupby("match_id")[col].agg(["sum","size"])
    s, n = d["sum"].values, d["size"].values
    idx = rng.integers(0, len(d), size=(B, len(d)))
    est = s[idx].sum(1)/n[idx].sum(1)*k
    return s.sum()/n.sum()*k, np.percentile(est,10), np.percentile(est,90), n.sum()
rows=[]
for lab,mk in sit.items():
    for side in ["bat","bowl"]:
        m,lo,hi,n = boot(side,mk)
        verdict = "real weakness" if hi<0 else ("real strength" if lo>0 else "within noise")
        rows.append((lab,side,n,round(m,2),round(lo,2),round(hi,2),verdict))
rr_gap = pd.DataFrame(rows,columns=["situation","side","RR_balls","gap","low80","high80","verdict"])
showtbl(rr_gap, "RR gap to the average team, with 80% band and verdict")
rr_gap.to_pickle(INTERIM+"rr_gap_boot.pkl")



situation,side,RR_balls,gap,low80,high80,verdict
Powerplay (1-6),bat,"2,749",17.17,8.68,25.31,real strength
Powerplay (1-6),bowl,"2,845",1.70,-0.57,4.05,within noise
Middle (7-15),bat,"4,033",-5.53,-10.83,-0.32,real weakness
Middle (7-15),bowl,"4,277",-1.22,-2.34,-0.06,real weakness
Death (16-20),bat,"1,966",-2.15,-8.68,4.54,within noise
Death (16-20),bowl,"2,250",-1.01,-2.77,0.76,within noise
50-50 (all overs),bat,"2,823",8.36,1.95,14.79,real strength
50-50 (all overs),bowl,"2,956",0.78,-0.97,2.60,within noise
Death in innings 1,bat,"1,134",-3.26,-11.64,4.86,within noise
Death in innings 1,bowl,"1,194",-0.68,-3.13,1.76,within noise


**Cell 3: Are the gaps real or just luck?**

**Procedure**
1. Resampled RR's matches with replacement 2,000 times. The match is the unit, because balls inside one match are linked.
2. Took the 10th–90th percentile range of the result as the **80% band**.
3. Called a gap **real** only if its whole band sits on one side of zero. Otherwise it is **within noise**.

**Observations**
- **Real strengths:** powerplay batting (+16.6, band +8.1 to +24.5), 50-50 batting (+8.7, band +2.7 to +14.8) and pressure batting (+11.1, band +2.6 to +19.1).
- **Real weaknesses:** middle-overs batting (−5.2, band −10.3 to −0.2) and middle-overs bowling (−1.4, band −2.5 to −0.2).
- **Within noise:** death batting (−1.4), death bowling (−1.05, band −2.8 to +0.7), 50-50 bowling, 50-50 death batting and bowling, and pressure bowling.
- **This partly supports the original view.** RR start well and handle 50-50 batting well. The statistically solid weakness is the **middle overs**, not the death.
- **Death-over weakness is directional but unproven.** The 2025–26 slide is consistent with it, but a five-season sample cannot separate it from bad luck.
- **How we phrase it in the report:** the data confirms a strong start and shows a middle-overs weakness, and the death-overs weakness is suggestive but not proven.

In [5]:
#%% ----- CELL 4 -----
# ============ CELL 4 : FINISHER VIEW (lower-order batters at the death) ============
# TECHNICAL: the pressure rating favours top-order batters (they face many pressure balls). A finisher faces few balls,
#            so we build a SEPARATE rating: batting position 5-8, overs 16-20, all T20 data 2022-26 (IPL + international),
#            net_bat per 100 balls (runs above expected, wickets converted to runs), shrunk toward the finisher average.
# FUNCTIONAL: "Who adds runs when the innings is being closed out, relative to what an average finisher would do in the same
#              situation?" Sample sizes are shown openly: a 60-ball finisher is a hint, not a verdict.
pak_ids = set(squads.loc[squads.team_name.str.contains("Pakistan", na=False),"player_id"])
excluded = set(pm.loc[pm.availability_guard.eq("EXCLUDED"),"player_id"])
fin = ball[(ball.phase=="death") & ball.batting_order.between(5,8) & ball.net_bat.notna()].copy()
fin["is_boundary_ball"] = (fin.is_four|fin.is_six).astype(int)
fin["legal"] = fin.is_legal.astype(int)

def shrink(raw, n, per_ball_sd):
    """empirical-Bayes with DerSimonian-Laird: separates real spread between players (tau2) from sampling noise (se2).
       trust B = tau2/(tau2+se2): 1 = take the player's own number, 0 = fall back to the finisher average."""
    se2 = per_ball_sd**2/n
    w = 1/se2; mu = np.sum(w*raw)/np.sum(w)
    Q = np.sum(w*(raw-mu)**2); k = len(raw)
    tau2 = max((Q-(k-1))/(np.sum(w)-np.sum(w**2)/np.sum(w)), 1e-8)
    B = tau2/(tau2+se2)
    return mu+B*(raw-mu), np.sqrt(B*se2), B

g = fin.groupby("striker_id").agg(balls=("net_bat","size"), net=("net_bat","sum"),
        runs=("runs","sum"), exp=("exp_bat_runs","sum"), bdry=("is_boundary_ball","mean"),
        outs=("y_bat_out","sum"), exp_outs=("p_bat_out","sum"),
        ipl_balls=("match_id", lambda s: s.isin(ipl_ids).sum()), last_year=("year","max"))
g = g[g.balls>=30].copy()
g["raw_per100"] = g.net/g.balls*100
sd_ball = fin.net_bat.std()
post, sd, B = shrink(g.raw_per100.values/100, g.balls.values, sd_ball)
g["rating"] = post*100; g["sd"] = sd*100; g["trust"] = B
g["low80"] = g.rating-1.28*g.sd; g["high80"] = g.rating+1.28*g.sd
g["sr"] = g.runs/g.balls*100; g["exp_sr"] = g.exp/g.balls*100
g["outs_vs_exp"] = g.outs-g.exp_outs
g["thin"] = np.where(g.balls<100,"THIN (<100 balls)","")
g = g.join(pm.set_index("player_id")[["display_name","age","current_team","availability_guard","role"]])
g["pakistan"] = g.index.isin(pak_ids)
g["recent"] = g.last_year>=2025
g["eligible"] = (~g.index.isin(excluded)) & (~g.pakistan) & g.recent
cols = ["display_name","age","current_team","balls","ipl_balls","sr","exp_sr","bdry","outs_vs_exp","raw_per100","rating","low80","high80","trust","thin"]
fin_tbl = g[g.eligible].sort_values("rating",ascending=False)[cols].round(2)
print("Finishers rated:", len(g), "| eligible (active 2025+, not excluded, not Pakistan):", int(g.eligible.sum()))
print("Average finisher: SR %.1f  | per-ball sd %.2f"%(fin.runs.sum()/len(fin)*100, sd_ball))
print("Shrinkage trust (B) median: %.2f  -> ratings lean heavily on the finisher average; ranks are tiers, not exact."%g.trust.median())
showtbl(fin_tbl.head(20), "Top 20 finishers (positions 5-8, death overs)")
fin_tbl.to_parquet(INTERIM+"rr_finishers.parquet")
showtbl(g[g.index.isin(rr_squad.player_id)].sort_values("rating",ascending=False)[cols].round(2), "RR's own finishers (2026 squad)")


Finishers rated: 187 | eligible (active 2025+, not excluded, not Pakistan): 136
Average finisher: SR 163.1  | per-ball sd 2.25
Shrinkage trust (B) median: 0.06  -> ratings lean heavily on the finisher average; ranks are tiers, not exact.


display_name,age,current_team,balls,ipl_balls,sr,exp_sr,bdry,outs_vs_exp,raw_per100,rating,low80,high80,trust,thin
Tim David,30.00,Royal Challengers Bengaluru,539,399,201.67,174.48,0.29,-7.43,31.66,12.41,5.34,19.48,0.32,
Mahendra Singh Dhoni,45.00,Chennai Super Kings,321,321,178.19,160.40,0.26,-10.26,28.16,8.73,1.15,16.32,0.22,
Rinku Singh,29.00,Kolkata Knight Riders,567,390,189.59,179.47,0.29,-14.80,18.60,8.35,1.34,15.37,0.34,
Marcus Stoinis,37.00,Punjab Kings,288,206,195.49,169.08,0.29,-1.24,27.80,8.21,0.53,15.88,0.20,
Rovman Powell,33.00,Kolkata Knight Riders,448,162,191.52,171.46,0.27,0.69,19.56,7.85,0.58,15.13,0.28,
Ashutosh Sharma,28.00,Delhi Capitals,223,217,192.83,168.11,0.29,-3.40,29.66,7.57,-0.29,15.43,0.17,
Azmatullah Omarzai,33.00,Punjab Kings,198,38,192.93,161.74,0.27,1.58,28.61,6.99,-0.94,14.92,0.15,
Romario Shepherd,32.00,Royal Challengers Bengaluru,345,101,178.55,166.81,0.26,-7.58,18.87,6.87,-0.65,14.39,0.23,
Ramandeep Singh,29.00,Kolkata Knight Riders,139,79,187.77,163.67,0.27,-5.16,36.15,6.81,-1.30,14.93,0.11,
Namboori Tilak Varma,24.00,Mumbai Indians,325,174,192.31,182.11,0.27,-8.09,18.27,6.57,-1.00,14.15,0.22,


display_name,age,current_team,balls,ipl_balls,sr,exp_sr,bdry,outs_vs_exp,raw_per100,rating,low80,high80,trust,thin
Donovan Ferreira,29.00,Rajasthan Royals,125,94,204.80,176.39,0.30,0.10,28.14,5.69,-2.47,13.84,0.10,
Dhruv Jurel,25.00,Rajasthan Royals,216,212,175.00,169.84,0.24,-3.26,10.05,4.30,-3.58,12.17,0.16,
Shubham Dubey,32.00,Rajasthan Royals,88,88,171.59,164.57,0.23,-0.81,10.01,3.68,-4.60,11.97,0.07,THIN (<100 balls)
Dasun Shanaka,35.00,Rajasthan Royals,321,24,165.73,158.16,0.24,2.29,5.25,3.65,-3.94,11.23,0.22,
Ravi Bishnoi,26.00,Rajasthan Royals,32,15,103.12,142.29,0.12,0.82,-47.45,1.79,-6.69,10.27,0.03,THIN (<100 balls)
Adam Milne,34.00,Rajasthan Royals,40,0,110.00,139.06,0.12,1.55,-41.66,1.65,-6.80,10.10,0.03,THIN (<100 balls)
Shimron Hetmyer,30.00,Rajasthan Royals,451,385,168.51,170.37,0.22,1.17,-2.70,1.50,-5.76,8.77,0.29,
Jofra Archer,31.00,Rajasthan Royals,49,40,134.69,151.02,0.16,3.48,-39.37,1.41,-7.01,9.83,0.04,THIN (<100 balls)
Sam Curran,28.00,Rajasthan Royals,259,111,154.44,165.57,0.19,2.87,-14.72,-0.16,-7.92,7.59,0.19,
Riyan Parag,25.00,Rajasthan Royals,131,118,144.27,160.09,0.18,5.53,-29.50,-0.22,-8.36,7.92,0.10,


**Cell 4: Finisher view (batting positions 5–8, overs 16–20)**

**Procedure**
1. The pressure rating favours top-order batters because they face many pressure balls. A finisher faces few balls, so this is a **separate rating**.
2. Used all T20 death balls from 2022–26 faced at positions 5–8, with at least 30 balls (187 batters). Rated each on net runs per 100 balls.
3. **Shrank small samples toward the finisher average** (DerSimonian-Laird). Trust is the share of the rating that comes from the player's own record (median about 0.06; most players sit at 0.1–0.35).
4. Kept only active (2025+), non-excluded, non-Pakistani players (136). Anyone with fewer than 100 balls is marked **THIN**.

**Observations**
- **Tim David is the clear standout:** +13.4 (band +6.2 to +20.6) on 539 balls, with 7.6 fewer dismissals than expected and a strike rate of 201.7 against 173 expected.
- **The next tier is Rinku Singh (+9.0), Dhoni (+8.9), Stoinis (+8.8) and Powell (+8.5).** Their bands only just clear zero.
- **Dhoni is 45.** That is an availability problem and is penalised in the auction layer.
- **RR's own finishers:** Ferreira is the best at +6.5, but his band spans zero and he has only 125 balls. Jurel is +4.6. Dubey is +4.0 on a THIN sample. Shanaka is +3.8, but only 24 of his 321 balls were in the IPL. Hetmyer is average (+2.0). Parag (−0.2), Curran (−0.1) and Jadeja (−2.3) rate slightly below the finisher average.
- **No RR player has a band clearly above zero.**
- **Scale note:** the finisher average is itself positive, because only players with 30+ death balls are included. Compare players with each other, not with zero.

In [6]:
#%% ----- CELL 5 -----
# ============ CELL 5 : DEATH-BOWLER VIEW (overs 16-20, both innings) ============
# TECHNICAL: net_bowl per 24 balls (= 4 overs) = runs saved vs expected incl. wickets converted to runs; same DL shrinkage as cell 4.
#            Execution columns (yorker/slower-ball share, dot%, boundary% conceded) explain HOW the value is created.
# FUNCTIONAL: "Who protects runs when batters are swinging?" and "does the bowler repeat a skill (yorkers, slower balls)
#              or just got lucky?" Wide uncertainty bands stay visible.
db = ball[(ball.phase=="death") & ball.net_bowl.notna()].copy()
db["bdry"] = (db.is_four|db.is_six).astype(int)
db["dot"]  = ((db.runs_conceded==0)&db.is_legal.astype(bool)).astype(int)
db["york"] = db.tag_delivery.eq("yorker").astype(int)
db["slow"] = db.tag_delivery.eq("slower ball / cutter").astype(int)
h = db.groupby("bowler_id").agg(balls=("net_bowl","size"), net=("net_bowl","sum"), runs=("runs_conceded","sum"),
        exp=("exp_runs_conc","sum"), wkts=("y_bowl_wkt","sum"), exp_w=("p_bowl_wkt","sum"),
        dot=("dot","mean"), bdry=("bdry","mean"), york=("york","mean"), slow=("slow","mean"),
        ipl_balls=("match_id", lambda s: s.isin(ipl_ids).sum()), last_year=("year","max"))
h = h[h.balls>=60].copy()
h["raw_per24"] = h.net/h.balls*24
sdb = db.net_bowl.std()
post, sd, B = shrink(h.raw_per24.values/24, h.balls.values, sdb)
h["rating"]=post*24; h["sd"]=sd*24; h["trust"]=B
h["low80"]=h.rating-1.28*h.sd; h["high80"]=h.rating+1.28*h.sd
h["econ"]=h.runs/h.balls*6; h["exp_econ"]=h.exp/h.balls*6; h["wkts_vs_exp"]=h.wkts-h.exp_w
h["thin"]=np.where(h.balls<100,"THIN (<100 balls)","")
h = h.join(pm.set_index("player_id")[["display_name","age","current_team","availability_guard","pace_or_spin"]])
h["eligible"] = (~h.index.isin(excluded)) & (~h.index.isin(pak_ids)) & (h.last_year>=2025)
cols_b = ["display_name","age","current_team","pace_or_spin","balls","ipl_balls","econ","exp_econ","wkts_vs_exp","dot","york","slow","raw_per24","rating","low80","high80","trust","thin"]
bow_tbl = h[h.eligible].sort_values("rating",ascending=False)[cols_b].round(2)
print("Death bowlers rated:", len(h), "| eligible:", int(h.eligible.sum()), "| trust median %.2f"%h.trust.median())
showtbl(bow_tbl.head(20), "Top 20 death-over bowlers")
bow_tbl.to_parquet(INTERIM+"rr_death_bowlers.parquet")
showtbl(h[h.index.isin(rr_squad.player_id)].sort_values("rating",ascending=False)[cols_b].round(2), "RR's own death bowlers (2026 squad)")

Death bowlers rated: 186 | eligible: 145 | trust median 0.03


display_name,age,current_team,pace_or_spin,balls,ipl_balls,econ,exp_econ,wkts_vs_exp,dot,york,slow,raw_per24,rating,low80,high80,trust,thin
Jasprit Bumrah,33.00,Mumbai Indians,Pace,566,405,7.20,9.60,0.81,0.35,0.16,0.24,9.83,1.42,0.34,2.50,0.10,
Sunil Narine,38.00,Kolkata Knight Riders,Spin,220,220,5.95,8.76,5.91,0.40,0.00,0.00,15.19,1.10,-0.01,2.21,0.04,
Matheesha Pathirana,24.00,Kolkata Knight Riders,Pace,605,444,8.31,9.38,7.45,0.33,0.12,0.21,6.07,1.08,0.01,2.16,0.10,
Rashid Khan,28.00,Gujarat Titans,Spin,528,327,8.26,8.69,18.05,0.37,0.00,0.00,6.75,1.08,-0.00,2.16,0.09,
Kuldeep Yadav,32.00,Delhi Capitals,Spin,303,239,7.98,9.15,2.86,0.35,0.00,0.00,6.05,0.81,-0.29,1.92,0.05,
Wanindu Hasaranga,29.00,Lucknow Super Giants,Spin,273,116,8.22,8.62,9.03,0.34,0.00,0.00,6.45,0.80,-0.30,1.91,0.05,
Lungi Ngidi,30.00,Delhi Capitals,Pace,256,121,8.70,9.91,2.54,0.36,0.04,0.60,6.34,0.78,-0.33,1.89,0.05,
Rasikh Salam,26.00,Royal Challengers Bengaluru,Pace,189,151,8.98,10.33,2.89,0.31,0.12,0.30,7.63,0.76,-0.36,1.87,0.03,
Gerald Coetzee,26.00,Sunrisers Hyderabad,Pace,212,106,8.72,9.99,2.28,0.32,0.06,0.26,6.67,0.75,-0.36,1.86,0.04,
Arshdeep Singh,27.00,Punjab Kings,Pace,"1,171",561,9.12,9.69,-4.05,0.30,0.16,0.12,1.81,0.75,-0.28,1.77,0.18,


display_name,age,current_team,pace_or_spin,balls,ipl_balls,econ,exp_econ,wkts_vs_exp,dot,york,slow,raw_per24,rating,low80,high80,trust,thin
Dasun Shanaka,35.00,Rajasthan Royals,Pace,109,6,8.97,9.55,4.79,0.28,0.08,0.30,8.76,0.68,-0.45,1.80,0.02,
Nandre Burger,31.00,Rajasthan Royals,Pace,88,69,8.18,9.06,0.86,0.41,0.09,0.23,4.95,0.58,-0.54,1.71,0.02,THIN (<100 balls)
Brijesh Sharma,24.00,Rajasthan Royals,Pace,103,103,9.55,10.00,1.09,0.33,0.07,0.36,3.33,0.56,-0.56,1.69,0.02,
Jofra Archer,31.00,Rajasthan Royals,Pace,367,232,9.50,9.73,0.45,0.31,0.09,0.20,1.10,0.55,-0.55,1.65,0.07,
Kwena Maphaka,20.00,Rajasthan Royals,Pace,75,14,10.00,10.42,-1.58,0.27,0.04,0.29,-1.43,0.48,-0.65,1.61,0.01,THIN (<100 balls)
Adam Milne,34.00,Rajasthan Royals,Pace,160,12,9.86,9.38,0.87,0.34,0.11,0.15,-1.12,0.46,-0.66,1.58,0.03,
Tushar Deshpande,31.00,Rajasthan Royals,Pace,392,380,10.39,10.11,2.40,0.28,0.12,0.25,-0.25,0.46,-0.64,1.55,0.07,
Sam Curran,28.00,Rajasthan Royals,Pace,459,190,9.83,9.91,-2.06,0.26,0.07,0.28,-0.35,0.44,-0.65,1.53,0.08,
Sandeep Sharma,33.00,Rajasthan Royals,Pace,365,365,10.44,10.35,-2.64,0.21,0.12,0.16,-1.41,0.38,-0.71,1.48,0.07,
Kuldeep Sen,30.00,Rajasthan Royals,Pace,65,65,12.55,10.16,-0.82,0.31,0.05,0.14,-11.43,0.36,-0.77,1.49,0.01,THIN (<100 balls)


**Cell 5: Death-bowler view (overs 16–20)**

**Procedure**
1. Rated bowlers with at least 60 death balls on **net runs saved per 24 balls**, using the same shrinkage as Cell 4.
2. Showed how the value is created: dot%, yorker share, slower-ball share, economy against expected, and wickets against expected.
3. Applied the same eligibility filters (145 eligible) and marked samples under 100 balls as THIN.

**Observations**
- **Four bowlers clear zero at 80%:** Bumrah +1.54 (band +0.38 to +2.69) on 566 balls, Pathirana +1.28, Narine +1.20 (band touches zero) and Rashid Khan +1.18.
- **Their economy beats expected by 0.5 to 2.8 runs per over.** Bumrah's is 7.20 against 9.53 expected.
- **Different skills create the value.** Bumrah and Arshdeep lean on yorkers (16% of balls). Pathirana relies on slower balls (21%). Narine and Rashid win with dots (0.40 and 0.37) and wickets.
- **RR's own bowlers sit between +0.14 and +0.69, and every band spans zero.** Their raw per-24 numbers are: Archer +1.43, Sandeep −1.51, Deshpande −0.60, Curran −0.41, Bishnoi −4.28, Kuldeep Sen −11.5 (THIN, 65 balls).
- **No RR bowler is a proven death specialist.** The highest yorker share among RR's bowlers is 12% (Sandeep, Deshpande).
- **Scale note:** every rating is positive because bowlers with 60+ death balls are above the league average. Use the raw per-24 column to see who actually saves runs.

**Summary: What the data says about RR**

1. **Strengths (proven):** powerplay batting, 50-50 batting and pressure batting.
2. **Weaknesses (proven):** middle-overs batting (10th of 10) and middle-overs bowling (9th of 10).
3. **Weaknesses (suggestive, not proven):** death batting and death bowling, which worsened in 2025–26. 50-50 death bowling ranks 10th on a small sample.
4. **Finisher view:** no RR finisher is clearly above the finisher average. Tim David is the clear outside standout, and Dhoni's age is an availability flag.
5. **Death-bowler view:** no RR bowler is a proven death specialist. Bumrah, Pathirana, Narine and Rashid Khan are the proven death bowlers outside RR.
6. **Next:** pool 2025–26 to test whether the death slide is real, build a middle-overs view, then combine with the retention list and prices (Layer 2).

In [7]:
#%% ----- CELL 6 -----
# ============ CELL 6 : (a) Is the 2025-26 death slide real?  (b) Where exactly do RR lose value in the middle overs? ============
# TECHNICAL: same match-level bootstrap as cell 3, but (a) restricted to seasons 2025-26 and (b) split by bowler type (pace vs spin).
# FUNCTIONAL: (a) the death slide looked real in the rankings; pooling the two recent seasons gives a fairer test than 5 seasons averaged.
#             (b) "weak in the middle overs" is too vague to shop for: we need to know if the fix is a spin-hitter, a pace-hitter,
#                 a spin bowler or a pace bowler.
def boot_sel(side, mk, B=2000, seed=11):
    rng = np.random.default_rng(seed)
    col, tid, k = ("net_bat","batting_team_id",100) if side=="bat" else ("net_bowl","bowling_team_id",24)
    d = ib[mk & (ib[tid]==RR) & ib[col].notna()].groupby("match_id")[col].agg(["sum","size"])
    if len(d)<8: return np.nan,np.nan,np.nan,int(d["size"].sum())
    s,n = d["sum"].values, d["size"].values
    idx = rng.integers(0,len(d),size=(B,len(d)))
    est = s[idx].sum(1)/n[idx].sum(1)*k
    return s.sum()/n.sum()*k, np.percentile(est,10), np.percentile(est,90), int(n.sum())
def verdict(m,lo,hi): return "n/a" if np.isnan(m) else ("real weakness" if hi<0 else ("real strength" if lo>0 else "within noise"))

recent = ib.season>=2025
rows=[]
for lab,mk in {"Death (16-20)":ib.is_death, "Death in innings 1":ib.is_death&ib.innings.eq(1),
               "Death in innings 2":ib.is_death&ib.innings.eq(2), "50-50 AND death":ib.fifty_death}.items():
    for side in ["bat","bowl"]:
        m,lo,hi,n = boot_sel(side, mk & recent)
        rows.append((lab,side,n,round(m,2),round(lo,2),round(hi,2),verdict(m,lo,hi)))
rec_tbl = pd.DataFrame(rows,columns=["situation","side","RR_balls","gap","low80","high80","verdict"]);
showtbl(rec_tbl, "(a) 2025-26 only: RR vs league")

mid = ib.phase.eq("middle")
rows=[]
for lab,mk in {"Middle bat vs PACE":mid&ib.pace_or_spin.eq("Pace"), "Middle bat vs SPIN":mid&ib.pace_or_spin.eq("Spin"),
               "Middle bowl: RR pace bowlers":mid&ib.pace_or_spin.eq("Pace"), "Middle bowl: RR spin bowlers":mid&ib.pace_or_spin.eq("Spin")}.items():
    side = "bat" if "bat" in lab else "bowl"
    m,lo,hi,n = boot_sel(side, mk)
    rows.append((lab,side,n,round(m,2),round(lo,2),round(hi,2),verdict(m,lo,hi)))
mid_tbl = pd.DataFrame(rows,columns=["split","side","RR_balls","gap","low80","high80","verdict"]);
showtbl(mid_tbl, "(b) Middle overs (7-15), all five seasons")

# who inside RR drives the middle-overs batting gap? (RR batters, middle overs, 150+ balls)
mb = ib[mid & (ib.batting_team_id==RR) & ib.net_bat.notna()].groupby("striker_id").agg(balls=("net_bat","size"), net=("net_bat","sum"))
mb["per100"]=mb.net/mb.balls*100
mb = mb[mb.balls>=150].join(pm.set_index("player_id").display_name).sort_values("per100")
mb["in_2026_squad"] = mb.index.isin(rr_squad.player_id)   # Ashwin, Padikkal, Samson, Buttler etc. may have left; flag who is still here
showtbl(mb.round(1), "RR batters in the middle overs (150+ balls, net runs per 100 balls)")
rec_tbl.to_parquet(INTERIM+"rr_death_recent.parquet"); mid_tbl.to_parquet(INTERIM+"rr_middle_split.parquet")


situation,side,RR_balls,gap,low80,high80,verdict
Death (16-20),bat,727,-10.03,-22.99,3.19,within noise
Death (16-20),bowl,877,-1.73,-4.32,1.04,within noise
Death in innings 1,bat,392,-11.32,-30.76,8.18,within noise
Death in innings 1,bowl,531,-4.02,-7.45,-0.88,real weakness
Death in innings 2,bat,335,-8.52,-24.19,6.67,within noise
Death in innings 2,bowl,346,1.78,-2.38,5.82,within noise
50-50 AND death,bat,173,9.19,-19.67,37.61,within noise
50-50 AND death,bowl,201,-4.61,-11.08,2.24,within noise


split,side,RR_balls,gap,low80,high80,verdict
Middle bat vs PACE,bat,"1,750",-8.56,-16.95,-0.25,real weakness
Middle bat vs SPIN,bat,"2,283",-3.22,-10.34,4.28,within noise
Middle bowl: RR pace bowlers,bowl,"1,546",-2.55,-4.58,-0.52,real weakness
Middle bowl: RR spin bowlers,bowl,"2,731",-0.47,-1.88,1.00,within noise


balls,net,per100,display_name,in_2026_squad
152,-35.30,-23.20,Ravichandran Ashwin,False
534,-114.20,-21.40,Yashasvi Jaiswal,True
279,-57.80,-20.70,Devdutt Padikkal,False
403,-21.80,-5.40,Dhruv Jurel,True
636,-23.00,-3.60,Sanju Samson,False
406,28.10,6.90,Jos Buttler,False
279,25.60,9.20,Shimron Hetmyer,True
642,90.60,14.10,Riyan Parag,True


**Cell 6: Is the 2025-26 death slide real, and where do RR lose value in the middle overs?**

**Procedure**
1. Repeated the match-level bootstrap from Cell 3 on the **2025 and 2026 seasons only**, to test whether the recent death-overs slide is real.
2. Split RR's middle-overs (7–15) gap by **bowler type** for batting (pace vs spin) and by **RR's own bowler type** for bowling.
3. Listed RR batters with 150+ middle-overs balls and flagged who is still in the 2026 squad.

**Observations**
- **The death-bowling weakness is real in 2025-26.** In innings 1 it is −4.07 net runs per 24 balls (band −7.35 to −1.05, 531 balls). Over both innings it is −1.90, within noise, because innings 2 is fine (+1.45).
- **50-50 death bowling is also a real weakness in 2025-26:** −6.02 (band −11.13 to −1.45), on a small sample of 202 balls.
- **Death batting is poor but unproven:** −7.79 per 100 balls over 727 balls, and the band includes zero. In innings 1 it is −10.05, still within noise.
- **RR's middle-overs bowling problem is in the pace bowlers:** −2.65 per 24 balls (band −4.62 to −0.72, 1,546 balls). The spin bowlers are within noise (−0.62).
- **The batting gap is not clearly pace or spin.** Against pace it is −8.12 (band −16.37 to +0.06, only just touching zero) and against spin −3.04 (within noise). Pace is the likelier culprit.
- **Players behind the middle-overs batting gap:**
  - Still in the 2026 squad: Jaiswal (−20.7 per 100, 534 balls) and Jurel (−4.9, 403 balls).
  - Players who have left: Ashwin, Padikkal and Samson also rate negative.
  - Hetmyer (+8.6) and Parag (+14.4, 642 balls) are positive.
- **Why Jaiswal looks weak here:** he is an opener, so his middle-overs balls come when he is still batting after the powerplay. Part of that is his role and not only his form.
- **Conclusion for the report:** two weaknesses now pass the noise test: **middle-overs pace bowling** and **innings-1 death bowling**. Both point to **pace bowling depth**, with death batting and middle-overs batting as secondary needs.

In [8]:

#%% ----- CELL 7 -----
# ============ CELL 7 : MIDDLE-OVERS VIEW (overs 7-15): best batters and bowlers outside the shortlist logic ============
# TECHNICAL: net_bat per 100 balls and net_bowl per 24 balls over overs 7-15, all T20 data 2022-26, DL-shrunk (shrink() from cell 4).
#            Batters are split by what they do against pace and against spin; bowlers by pace/spin and by economy vs expected.
# FUNCTIONAL: RR's proven weakness is here. We want batters who keep scoring AND do not get out when no one is "finishing",
#             and bowlers who control the middle (dots, wickets) instead of leaking runs.
mb_all = ball[(ball.phase=="middle") & ball.net_bat.notna()].copy()
g = mb_all.groupby("striker_id").agg(balls=("net_bat","size"), net=("net_bat","sum"), runs=("runs","sum"), exp=("exp_bat_runs","sum"),
        outs=("y_bat_out","sum"), exp_outs=("p_bat_out","sum"), last_year=("year","max"),
        ipl_balls=("match_id", lambda s: s.isin(ipl_ids).sum()))
for t in ["Pace","Spin"]:
    x = mb_all[mb_all.pace_or_spin==t].groupby("striker_id").agg(**{f"balls_{t}":("net_bat","size"), f"net_{t}":("net_bat","sum")})
    g = g.join(x)
    g[f"per100_{t}"] = np.where(g[f"balls_{t}"]>=60, g[f"net_{t}"]/g[f"balls_{t}"]*100, np.nan)
g = g[g.balls>=150].copy()
g["raw_per100"]=g.net/g.balls*100
post,sd,B = shrink(g.raw_per100.values/100, g.balls.values, mb_all.net_bat.std())
g["rating"]=post*100; g["sd"]=sd*100; g["trust"]=B; g["low80"]=g.rating-1.28*g.sd; g["high80"]=g.rating+1.28*g.sd
g["sr"]=g.runs/g.balls*100; g["exp_sr"]=g.exp/g.balls*100; g["outs_vs_exp"]=g.outs-g.exp_outs
g = g.join(pm.set_index("player_id")[["display_name","age","current_team","role"]])
g["eligible"]=(~g.index.isin(excluded))&(~g.index.isin(pak_ids))&(g.last_year>=2025)
cb=["display_name","age","current_team","balls","ipl_balls","sr","exp_sr","outs_vs_exp","per100_Pace","per100_Spin","raw_per100","rating","low80","high80","trust"]
mid_bat = g[g.eligible].sort_values("rating",ascending=False)[cb].round(2)
showtbl(mid_bat.head(15), "Middle-overs batters, top 15 (eligible: %d)"%len(mid_bat))
mid_bat.to_parquet(INTERIM+"rr_middle_batters.parquet")

mbw = ball[(ball.phase=="middle") & ball.net_bowl.notna()].copy()
mbw["dot"]=((mbw.runs_conceded==0)&mbw.is_legal.astype(bool)).astype(int)
h = mbw.groupby("bowler_id").agg(balls=("net_bowl","size"), net=("net_bowl","sum"), runs=("runs_conceded","sum"), exp=("exp_runs_conc","sum"),
        wkts=("y_bowl_wkt","sum"), exp_w=("p_bowl_wkt","sum"), dot=("dot","mean"), last_year=("year","max"),
        ipl_balls=("match_id", lambda s: s.isin(ipl_ids).sum()))
h = h[h.balls>=150].copy(); h["raw_per24"]=h.net/h.balls*24
post,sd,B = shrink(h.raw_per24.values/24, h.balls.values, mbw.net_bowl.std())
h["rating"]=post*24; h["sd"]=sd*24; h["trust"]=B; h["low80"]=h.rating-1.28*h.sd; h["high80"]=h.rating+1.28*h.sd
h["econ"]=h.runs/h.balls*6; h["exp_econ"]=h.exp/h.balls*6; h["wkts_vs_exp"]=h.wkts-h.exp_w
h = h.join(pm.set_index("player_id")[["display_name","age","current_team","pace_or_spin"]])
h["eligible"]=(~h.index.isin(excluded))&(~h.index.isin(pak_ids))&(h.last_year>=2025)
cw=["display_name","age","current_team","pace_or_spin","balls","ipl_balls","econ","exp_econ","wkts_vs_exp","dot","raw_per24","rating","low80","high80","trust"]
mid_bowl = h[h.eligible].sort_values("rating",ascending=False)[cw].round(2)
showtbl(mid_bowl.head(15), "Middle-overs bowlers, top 15 (eligible: %d)"%len(mid_bowl))
mid_bowl.to_parquet(INTERIM+"rr_middle_bowlers.parquet")

display_name,age,current_team,balls,ipl_balls,sr,exp_sr,outs_vs_exp,per100_Pace,per100_Spin,raw_per100,rating,low80,high80,trust
Suryakumar Yadav,36.00,Mumbai Indians,"1,744",937,165.31,146.29,-22.70,30.40,28.29,29.18,21.22,14.90,27.54,0.67
Heinrich Klaasen,35.00,Sunrisers Hyderabad,921,783,158.74,142.43,-18.34,16.53,44.52,31.85,18.91,11.27,26.56,0.52
Finn Allen,27.00,Kolkata Knight Riders,228,50,211.84,156.46,-0.93,14.94,86.00,58.57,16.28,6.51,26.05,0.21
Rajat Patidar,33.00,Royal Challengers Bengaluru,603,603,174.30,145.67,-2.53,17.87,46.66,31.91,16.13,7.70,24.56,0.41
Tim David,30.00,Royal Challengers Bengaluru,592,218,153.72,130.50,-4.42,23.41,35.15,29.04,14.84,6.38,23.31,0.41
Shivam Dube,33.00,Chennai Super Kings,"1,028",749,147.18,134.40,-11.43,14.99,28.92,21.45,13.99,6.57,21.41,0.54
Mark Chapman,32.00,Not in IPL 2026,544,0,152.21,132.84,-5.35,49.49,14.29,27.04,13.58,4.98,22.19,0.39
Devon Conway,35.00,Not in IPL 2026,638,342,148.28,143.45,-15.86,16.50,29.04,24.23,13.23,4.90,21.56,0.43
Shubman Gill,27.00,Gujarat Titans,"1,145",935,154.24,153.52,-25.60,0.63,27.79,18.16,12.55,5.34,19.75,0.57
Vaibhav Sooryavanshi,15.00,Rajasthan Royals,214,146,228.50,169.05,5.70,42.96,35.46,38.69,11.77,1.93,21.61,0.20


display_name,age,current_team,pace_or_spin,balls,ipl_balls,econ,exp_econ,wkts_vs_exp,dot,raw_per24,rating,low80,high80,trust
Varun Chakravarthy,35.00,Kolkata Knight Riders,Spin,"1,497",973,7.69,8.16,18.67,0.37,4.95,2.84,1.18,4.51,0.52
Patrick Cummins,33.00,Sunrisers Hyderabad,Pace,406,323,7.23,8.78,4.46,0.37,8.94,2.46,0.35,4.58,0.23
Kuldeep Yadav,32.00,Delhi Capitals,Spin,"1,644","1,178",7.74,8.11,15.65,0.33,3.81,2.33,0.70,3.95,0.55
Mujeeb Ur Rahman,25.00,Not in IPL 2026,Spin,333,7,6.77,7.13,11.00,0.39,9.57,2.31,0.15,4.47,0.20
Mustafizur Rahman,31.00,Kolkata Knight Riders,Pace,456,79,6.53,7.90,2.92,0.39,7.06,2.17,0.08,4.26,0.25
Jasprit Bumrah,33.00,Mumbai Indians,Pace,426,346,6.59,8.79,-3.05,0.39,7.03,2.08,-0.03,4.18,0.24
Krunal Pandya,35.00,Royal Challengers Bengaluru,Spin,959,959,7.14,8.02,1.57,0.30,3.91,1.93,0.08,3.77,0.41
Jamie Overton,32.00,Chennai Super Kings,Pace,291,112,7.01,8.10,4.61,0.40,8.25,1.89,-0.30,4.08,0.18
Wanindu Hasaranga,29.00,Lucknow Super Giants,Spin,"1,642",608,7.66,7.81,15.76,0.34,2.95,1.85,0.23,3.47,0.55
Jason Holder,35.00,Gujarat Titans,Pace,757,345,7.57,8.30,3.97,0.30,4.20,1.84,-0.09,3.77,0.36


**Cell 7: Middle-overs view (overs 7–15)**

**Procedure**
1. Rated every eligible batter (150+ middle-overs balls) and bowler (150+ balls) on net runs above expected: per 100 balls for batters and per 24 balls for bowlers. Used all T20 data from 2022–26.
2. **Shrank small samples toward the group average** (same method as Cells 4 and 5). Showed pace and spin splits for batters.
3. Kept only active (2025+), non-excluded, non-Pakistani players (batters 135 eligible, bowlers 134 eligible).

**Observations (bowlers)**
- **Varun Chakravarthy is the best middle-overs bowler at +2.75** (band +1.10 to +4.39) on 1,497 balls, with 18.9 more wickets than expected.
- **Kuldeep Yadav (+2.13, 1,644 balls, band +0.53 to +3.74) and Hasaranga (+1.84)** have the largest samples. Their trust is about 0.54, so their ratings come mostly from their own record.
- **Pace bowlers who clear zero at 80%:**
  - Cummins: +2.39 (band +0.31 to +4.48), economy 7.21 against 8.71 expected.
  - Mustafizur: +2.17 (band +0.11 to +4.23), on only 79 IPL balls.
- **Bumrah (+2.00) and Pathirana (+1.62)** also rate well here, so both are strong in the middle overs and the death. Narine (+1.64) is strong in both phases too.
- **Pathirana's band includes zero** (−0.37), so treat him as a promising tier, not a proven one.
- **Mujeeb Ur Rahman (+2.32) is not in an IPL 2026 squad** and has only 7 IPL balls, so his record is from other leagues. Treat that carefully.
- **Noor Ahmad (+1.72) is the youngest strong name at 21.**

**Observations (batters)**
- **Suryakumar Yadav is the best middle-overs batter at +21.0 per 100 balls** (band +14.9 to +27.2) on 1,744 balls. His trust is 0.67, so the rating comes mostly from his own record. He is strong against both pace (about +31) and spin (+27.8), and he gets out 24 times fewer than expected.
- **Klaasen (+18.0, band +10.6 to +25.5)** is an elite spin-hitter at +42.5 against spin and about +16 against pace.
- **Finn Allen (+15.7, band +6.2 to +25.2)** scores at a strike rate of 211.8 against 156 expected, but only 228 balls and 50 IPL balls back it. Treat his +86 against spin with caution.
- **Patidar (+15.5, band +7.3 to +23.8)** has the full 603 balls in the IPL and rates well against spin (+45.3) and pace (about +17).
- **Tim David (+14.4, band +6.2 to +22.7)** is strong in the middle overs as well as the death, which makes him the most versatile batter on the list.
- **Shivam Dube (+13.7, 1,028 balls)** is a consistent middle-overs hitter with high trust (0.54).
- **Vaibhav Sooryavanshi (+12.0, band +2.4 to +21.6)** is already in RR. He is notably strong against pace (about +46), but he gets out 5.3 times more than expected, so the high-risk style shows up here.
- **Dewald Brevis (+10.8, band +2.0 to +19.6)** is 23 and rates well on 409 balls. He is strongest against spin (+36.6) and weaker against pace (about +7), so he is a younger, narrower option.
- **Mark Chapman, Devon Conway and Harry Brook** (+9.8 to +13.7) are not in IPL 2026 squads, so they may be auction candidates. Their IPL samples are small (0, 342 and 45 balls).
- **Shubman Gill (+11.6), Lokesh Rahul (+11.3), Buttler (+10.7) and Shreyas Iyer (+10.1)** are solid but sit below the leaders. All four get out fewer times than expected (−16 to −25).

**Scale note:** as with the other views, the average rating is positive (batters and bowlers with 150+ middle-overs balls are better than average), so compare players with each other and not with zero.